## Mounting ADLS Gen2 Storage by OAuth 

In [0]:
# Widget Inputs
dbutils.widgets.text("container", "")
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("mount_name", "")

container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")
mount_name = dbutils.widgets.get("mount_name")

# Secret Scope Configuration (fixed names)
tenant_id = dbutils.secrets.get(scope="projectscope", key="tenant-id")
client_id = dbutils.secrets.get(scope="projectscope", key="client-id")
appsecret = dbutils.secrets.get(scope="projectscope", key="appsecret")

configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": appsecret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

mount_point = f"/mnt/{mount_name}"
source_uri = f"abfss://{container}@{storage_account}.dfs.core.windows.net/"

# Check if already mounted
if mount_point in [mount.mountPoint for mount in dbutils.fs.mounts()]:
    dbutils.notebook.exit(f"{mount_point} is already mounted.")
else:
    dbutils.fs.mount(
        source=source_uri,
        mount_point=mount_point,
        extra_configs=configs
    )
    dbutils.notebook.exit(f"Mounted {source_uri} to {mount_point}")
